# Track 4 · 01 emg2pose prepare — a balanced sample of the warm-up corpus onto Drive

**One job.** Put a small, deliberately balanced sample of emg2pose on Drive: harness-made
5-second windows of wrist EMG with their 20 joint-angle trajectories, about 2.5 GB in all.

Self-contained: upload this one file.

---

## The task, from source

`tracks/emg_pose` and `neuralbench/tasks/emg/pose/config.yaml`: 16-channel wrist EMG at
2 kHz in, 20 hand-joint angles (radians) out, over **5-s windows tiled every 5 s**. Scored
by mean angle error in degrees. **No preprocessing at all** for EMG: no filter, no scaling.
Recordings are read through `neuralfetch`'s `Salter2024Emg2pose` (via `mne_bids`), which
blanks the angles wherever the motion-capture solver failed (`BAD_IK`); any window touching
a blank is dropped. On a sample recording that kept 4 of 12 windows.

## Who is scored, and what we leave out

From the upstream `emg2pose_metadata.csv` the study joins on: the warm-up test is **20
held-out people doing 6 held-out movement types** (`split == test`,
`generalization == user_stage`, 7.5 h). **Every recording of those 20 people is excluded.**

The pool is everyone else: the 158 training people and the 15 validation people, across
all 29 movement types and both hands. One choice to know about: the training people's
recordings of the 6 held-out movement types (labelled `test`, `generalization == stage`,
not scored) are **included** for movement-type diversity. That is public data from other
people, but it means the warm-up's "unseen movement" half is no longer unseen for us; its
"unseen person" half stays clean. Set `INCLUDE_HELD_OUT_STAGES = False` to drop them.

## Balanced sampling under a 2.5 GB budget

The corpus is 370 h; the budget holds roughly 17 h. The warm-up and sealed tests are on
unseen people, so the sample is spread for diversity rather than depth:

- **People:** every eligible person gets the same share of windows.
- **Movement types:** each person's recordings are taken in rotation across types, least
  sampled overall first, so all 29 types end up about equally represented.
- **Hands:** left and right alternate within each person.
- **Within a recording:** at most `WINDOWS_PER_RECORDING` usable windows, spread across it,
  so a person's share covers many recordings rather than a few long ones.

The 15 validation people are kept as group `val` (unseen people, the right yardstick for
choosing epochs); everyone else is `train`.

## Storage, verified rather than assumed

- **Angles at 200 Hz** instead of 2 kHz. The harness itself stretches lower-rate
  predictions to full rate (nearest neighbour); on a real recording, angles held at 200 Hz
  differ from the 2 kHz truth by 0.12° on average, against scores of 12–22°.
- **EMG at 1 kHz** (`EMG_SFREQ`), resampled per window with the same pure-torch function
  the submission uses on its 2 kHz input, so training and scoring see identical data.
  Surface EMG carries little above 500 Hz. Set `EMG_SFREQ = 2000` to keep it native, at
  half the windows for the same budget.
- **EMG in microvolts.** In volts its sd is about 3e-5, too small for float16; the
  submission applies the same factor.

## Licence

emg2pose is **CC-BY-NC-SA-4.0** (non-commercial, share-alike). Fine for the competition;
flag before any other use. Cite Salter et al., NeurIPS 2024.

## Cost

Roughly 40–50 GB downloaded through temporary disk (only about a third of each recording
survives the label-failure rule), about 2.5 GB written to Drive. Likely 1–2 hours,
mostly download. It resumes: people already on Drive are skipped.

## Setup

In [ ]:
ROOT = '/content/drive/MyDrive/Neurips26'

BUDGET_GB = 2.5
EMG_SFREQ = 1000.0           # stored EMG rate; 2000 keeps it native (half the windows)
TARGET_SFREQ = 200.0         # stored joint-angle rate
WINDOWS_PER_RECORDING = 4    # at most this many usable windows from one recording
INCLUDE_HELD_OUT_STAGES = True
WORKERS = 8                  # people processed in parallel
SEED = 33

# The harness's settings (emg/pose task config). Do not tune these.
NATIVE_SFREQ = 2000.0
DURATION = 5.0
STRIDE = 5.0
EMG_SCALE = 1e6              # volts -> microvolts for storage; the solver applies the same

DEPOSIT = 'https://data.nemar.org/nm000281/v1.0.2'
METADATA_URL = 'https://fb-ctrl-oss.s3.amazonaws.com/emg2pose/emg2pose_metadata.csv'

print('NOTEBOOK VERSION: track4_01 v1')

In [ ]:
%pip install -q mne mne-bids

import importlib
for mod in ('mne', 'mne_bids', 'torch', 'numpy', 'pandas'):
    print(f'{mod:9} {importlib.import_module(mod).__version__}')

In [ ]:
import io, json, math, shutil, threading, time, urllib.request, warnings
warnings.filterwarnings('ignore')
from collections import Counter
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path
import numpy as np, pandas as pd, torch, mne, mne_bids
mne.set_log_level('ERROR')

if ROOT.startswith('/content/drive'):
    from google.colab import drive
    drive.mount('/content/drive')

OUT = Path(ROOT) / 'Track4' / 'emg2pose'
WIN = OUT / 'windows'
WIN.mkdir(parents=True, exist_ok=True)
TMP = Path('/content/emg2pose_tmp') if Path('/content').exists() else OUT.parent / '_emg_tmp'
TMP.mkdir(parents=True, exist_ok=True)


def get(rel, timeout=180):
    url = rel if rel.startswith('http') else f'{DEPOSIT}/{rel}' if rel else f'{DEPOSIT}/'
    for attempt in range(4):
        try:
            with urllib.request.urlopen(url, timeout=timeout) as r:
                return r.read()
        except Exception as e:
            if attempt == 3:
                raise RuntimeError(f'{rel}: {e}')
            time.sleep(5 * (attempt + 1))


def listing(rel):
    return json.loads(get(rel + '/' if rel else ''))['children']


# BIDS root skeleton that mne_bids needs to read a recording the way the harness does.
for f in ('dataset_description.json', 'participants.tsv'):
    (TMP / f).write_bytes(get(f))
meta = pd.read_csv(io.BytesIO(get(METADATA_URL)))
meta = meta.set_index('filename')

subjects = sorted(c['name'] for c in listing('') if c['kind'] == 'dir' and c['name'].startswith('sub-'))


def scans_of(sub):
    rows = []
    for ses in sorted(c['name'] for c in listing(sub) if c['kind'] == 'dir'):
        t = pd.read_csv(io.BytesIO(get(f'{sub}/{ses}/{sub}_{ses}_scans.tsv')), sep='\t')
        t['sub'], t['ses'] = sub, ses
        rows.append(t)
    return pd.concat(rows) if rows else pd.DataFrame()


with ThreadPoolExecutor(16) as pool:
    scans = pd.concat(list(pool.map(scans_of, subjects)), ignore_index=True)
scans['source'] = scans['source_file'].str.removesuffix('.hdf5')
recs = scans.join(meta[['user', 'split', 'generalization']], on='source')
print(f'{len(subjects)} people, {len(recs):,} recordings, {recs.duration.sum() / 3600:.0f} h '
      f'(the study expects 25,253 recordings)')

## Step 1 — the sampling plan

Decided before any download, from the metadata alone, so it is the same on every run.

In [ ]:
test_users = set(recs.loc[(recs.split == 'test') & (recs.generalization == 'user_stage'), 'user'])
pool_ = recs[~recs.user.isin(test_users) & (recs.duration >= DURATION)]
if not INCLUDE_HELD_OUT_STAGES:
    pool_ = pool_[pool_.split != 'test']
pool_ = pool_.assign(group=np.where(pool_.split == 'val', 'val', 'train'))

emg_steps = int(round(DURATION * EMG_SFREQ))
tgt_steps = int(round(DURATION * TARGET_SFREQ))
bytes_per_window = 2 * (16 * emg_steps + 20 * tgt_steps)            # float16
total_windows = int(BUDGET_GB * 1e9 / bytes_per_window)
people = sorted(pool_['sub'].unique())
quota = total_windows // len(people)

def k_dur(r):
    return float(r.duration)


rng = np.random.default_rng(SEED)
stage_use = Counter()
plan = {}
for sub in rng.permutation(people):
    mine = pool_[pool_['sub'] == sub]
    cells = {}
    for _, r in mine.iterrows():
        cells.setdefault((r.stage, r.side), []).append(r)
    for v in cells.values():
        rng.shuffle(v)
    order, need, last_side = [], quota, None
    while need > -0.5 * quota and any(cells.values()):   # plan 1.5x the quota: yields vary
        # least-sampled movement type first; alternate hands within the person
        open_cells = [k for k, v in cells.items() if v]
        open_cells.sort(key=lambda k: (stage_use[k[0]], k[1] == last_side, rng.random()))
        k = open_cells[0]
        order.append(cells[k].pop())
        stage_use[k[0]] += 1
        last_side = k[1]
        # expected usable windows: about a third of the tiled ones survive BAD_IK
        need -= min(WINDOWS_PER_RECORDING, max(1.0, k_dur(order[-1]) / DURATION / 3))
    plan[sub] = order

print(f'eligible people: {len(people)} ({sum(pool_.groupby("sub").group.first() == "val")} val), '
      f'excluded test people: {len(test_users)}')
print(f'budget {BUDGET_GB} GB -> {total_windows:,} windows of {bytes_per_window / 1e3:.0f} KB '
      f'({total_windows * DURATION / 3600:.1f} h), {quota} per person')
print(f'planned recordings: {sum(len(v) for v in plan.values()):,}, '
      f'movement types covered: {len(stage_use)}, '
      f'per-type range {min(stage_use.values())}-{max(stage_use.values())}')

## Step 2 — download, window and store, one person at a time

Each person's planned recordings are fetched until their quota of usable windows is met.
Windows come from neuralfetch's loading rules and the task's tiling; EMG is then resampled
and scaled exactly as the submission will do it.

In [ ]:
def emg_windows(bdf_path):
    """One recording -> EMG and joint-angle windows, as neuralfetch's Salter2024Emg2pose
    loads it and the emg/pose task cuts it: crop after the last labelled span, blank
    BAD_IK samples, tile 5-s windows every 5 s, drop any window touching a blank."""
    raw = mne_bids.read_raw_bids(mne_bids.get_bids_path_from_fname(bdf_path),
                                 verbose=False).load_data()
    valid_end = max((a['onset'] + a['duration'] for a in raw.annotations
                     if a['description'] != 'BAD_IK'),
                    default=raw.times[-1] + 1 / raw.info['sfreq'])
    stop = max(1, min(raw.n_times, raw.time_as_index([valid_end], use_rounding=True)[0]))
    if stop < raw.n_times:
        raw.crop(tmax=raw.times[stop - 1])
    bad = np.zeros(raw.n_times, dtype=bool)
    for a in raw.annotations:
        if a['description'] == 'BAD_IK':
            s, e = raw.time_as_index([a['onset'], a['onset'] + a['duration']], use_rounding=True)
            bad[max(0, s):min(raw.n_times, e)] = True
    emg = raw.copy().pick('emg').get_data()
    ang = raw.copy().pick('misc').get_data()
    sf = raw.info['sfreq']
    n_times = int(round(DURATION * sf))
    span = (raw.n_times / sf - DURATION)
    n = math.floor(span / STRIDE + 1e-9) + 1 if span >= -1e-10 else 0
    idx = np.round(np.arange(n) * STRIDE * sf).astype(int)
    keep = np.array([not bad[i:i + n_times].any() for i in idx], dtype=bool)
    idx = idx[keep]
    X = np.stack([emg[:, i:i + n_times] for i in idx]) if len(idx) else np.zeros((0, emg.shape[0], n_times))
    Y = np.stack([ang[:, i:i + n_times] for i in idx]) if len(idx) else np.zeros((0, ang.shape[0], n_times))
    return X.astype('float32'), Y.astype('float32'), idx / sf, dict(n_tiled=int(n), n_kept=int(len(idx)),
            channels=list(raw.copy().pick('emg').ch_names), joints=list(raw.copy().pick('misc').ch_names), sfreq=sf)


def resample_fft(x, n_out):
    """Same function as the submission's: resample along time via the rfft spectrum."""
    n_in = x.shape[-1]
    if n_in == n_out:
        return x
    spec = torch.fft.rfft(x.float(), dim=-1)
    out = torch.zeros(*spec.shape[:-1], n_out // 2 + 1, dtype=spec.dtype, device=x.device)
    keep = min(spec.shape[-1], out.shape[-1])
    out[..., :keep] = spec[..., :keep]
    return torch.fft.irfft(out, n=n_out, dim=-1) * (n_out / n_in)


def nearest(seq, n):
    """benchmark_utils.data.resample_labels: the harness's own nearest-neighbour rule."""
    L = seq.shape[-1]
    i = np.clip(np.floor(np.arange(n) * (L / n)).astype(int), 0, L - 1)
    return seq[..., i]


def fetch_recording(r):
    base = f'{r["sub"]}/{r["ses"]}'
    stem = r['filename'].split('/')[-1].removesuffix('_emg.bdf')
    d = TMP / base / 'emg'
    d.mkdir(parents=True, exist_ok=True)
    (TMP / base / f'{r["sub"]}_{r["ses"]}_scans.tsv').write_bytes(get(f'{base}/{r["sub"]}_{r["ses"]}_scans.tsv'))
    for suffix in ('_channels.tsv', '_emg.json', '_events.tsv', '_emg.bdf'):
        (d / f'{stem}{suffix}').write_bytes(get(f'{base}/emg/{stem}{suffix}'))
    return d / f'{stem}_emg.bdf'


def do_person(sub):
    dest = WIN / f'{sub}.npz'
    if dest.exists():
        return 'kept', json.loads(str(np.load(dest)['info']))
    Xs, Ys, rows, used = [], [], [], 0
    for r in plan[sub]:
        if used >= quota:
            break
        path = fetch_recording(r)
        try:
            X, Y, starts, info = emg_windows(path)
        finally:
            shutil.rmtree(path.parent, ignore_errors=True)
        if not len(X):
            continue
        take = np.unique(np.linspace(0, len(X) - 1, min(WINDOWS_PER_RECORDING, len(X), quota - used)).round().astype(int))
        x = resample_fft(torch.from_numpy(X[take]), emg_steps).numpy() * EMG_SCALE
        Xs.append(x.astype('float16')); Ys.append(nearest(Y[take], tgt_steps).astype('float16'))
        for t in take:
            rows.append(dict(recording=r['filename'].split('/')[-1], stage=r['stage'], side=r['side'],
                             start=float(starts[t]), split=r['split'], generalization=r['generalization']))
        used += len(take)
    info = dict(sub=sub, user=str(pool_.loc[pool_['sub'] == sub, 'user'].iloc[0]),
                group=str(pool_.loc[pool_['sub'] == sub, 'group'].iloc[0]),
                n_windows=used, recordings=len({x['recording'] for x in rows}))
    if not used:
        np.savez(dest, info=json.dumps(info))
        return 'empty', info
    np.savez(dest, X=np.concatenate(Xs), Y=np.concatenate(Ys),
             windows=json.dumps(rows), info=json.dumps(info))
    return 'new', info


t0 = time.time()
done, failed = {}, {}
with ThreadPoolExecutor(WORKERS) as pool:
    futs = {pool.submit(do_person, s): s for s in people}
    for i, f in enumerate(as_completed(futs), 1):
        s = futs[f]
        try:
            status, info = f.result()
            done[s] = info
            if i % 10 == 0 or i == len(people):
                print(f'  {i:3d}/{len(people)}  {s} {status}  {info["n_windows"]} windows  '
                      f'{time.time() - t0:.0f}s')
        except Exception as e:
            failed[s] = repr(e)
            print(f'  FAILED {s}: {e}')
print(f'\n{len(done)} people processed, {len(failed)} failed, {time.time() - t0:.0f}s')
if failed:
    print('Rerun this cell: finished people are kept and only the failures are retried.')

## Step 3 — balance report, manifest and checks

In [ ]:
rows, per_group, problems = [], Counter(), []
for f in sorted(WIN.glob('sub-*.npz')):
    z = np.load(f)
    info = json.loads(str(z['info']))
    if 'X' not in z:
        continue
    X, Y = z['X'], z['Y']
    if X.shape[1:] != (16, emg_steps) or Y.shape[1:] != (20, tgt_steps):
        problems.append(f'{f.name}: shapes {X.shape} {Y.shape}')
    if not (np.isfinite(X.astype('float32')).all() and np.isfinite(Y.astype('float32')).all()):
        problems.append(f'{f.name}: non-finite values')
    for w in json.loads(str(z['windows'])):
        rows.append(dict(w, sub=info['sub'], group=info['group']))
    per_group[info['group']] += len(X)
w = pd.DataFrame(rows)
if len(w):
    by_person = w.groupby('sub').size()
    by_stage = w.groupby('stage').size()
    print(f'windows: {len(w):,} ({len(w) * DURATION / 3600:.1f} h)  train {per_group["train"]:,}  '
          f'val {per_group["val"]:,}')
    print(f'people: {w["sub"].nunique()}  windows per person min {by_person.min()} '
          f'median {int(by_person.median())} max {by_person.max()}')
    print(f'movement types: {w.stage.nunique()}  windows per type min {by_stage.min()} '
          f'median {int(by_stage.median())} max {by_stage.max()}')
    print(f'hands: {w.side.value_counts().to_dict()}   recordings used: {w.recording.nunique():,}')
    if set(w['sub']) & set(recs.loc[recs.user.isin(test_users), 'sub']):
        problems.append('a scored test person is in the sample')
print('checks:', 'all passed' if not problems else problems[:10])

(OUT / 'manifest.json').write_text(json.dumps(dict(
    source='emg2pose (NEMAR nm000281 v1.0.2, neuralfetch Salter2024Emg2pose)',
    licence='CC-BY-NC-SA-4.0',
    harness=dict(native_sfreq=NATIVE_SFREQ, duration=DURATION, stride=STRIDE,
                 preprocessing='none', drop_windows_touching='BAD_IK'),
    stored=dict(emg_sfreq=EMG_SFREQ, emg_scale=EMG_SCALE, emg_unit='microvolt',
                target_sfreq=TARGET_SFREQ, target_unit='radian', dtype='float16',
                emg_resample='rfft (same as the solver)', target_resample='nearest (harness rule)'),
    sampling=dict(budget_gb=BUDGET_GB, windows_per_recording=WINDOWS_PER_RECORDING,
                  include_held_out_stages=INCLUDE_HELD_OUT_STAGES, seed=SEED, quota=quota),
    excluded_test_users=sorted(test_users),
    people={s: d for s, d in done.items()}, failed=failed), indent=1))
size = sum(f.stat().st_size for f in WIN.glob('*.npz')) / 1e9
print(f'\nmanifest -> {OUT / "manifest.json"}   windows on Drive: {size:.2f} GB')
print('READY for track4_02' if not failed and not problems and len(w) else
      'NOT READY: see the counts and checks above')